In [ ]:
# Install the required packages:
# - langchain: core LangChain framework
# - langchain-anthropic: LangChain's integration with Anthropic's Claude models
!pip install -q langchain langchain-anthropic


In [ ]:
from google.colab import userdata

# LangChain message types:
# AIMessage    — a response generated by the AI model
# HumanMessage — a message from the user
# SystemMessage — a standing instruction that shapes the model's behavior (not shown to the end user)
from langchain.messages import AIMessage, HumanMessage, SystemMessage

# ChatAnthropic: LangChain's wrapper around the Anthropic Claude API
from langchain_anthropic import ChatAnthropic

# Helpers for building multi-modal message content blocks (text + images)
from langchain_core.messages.content import create_image_block, create_text_block

# BaseModel: Pydantic base class used to define structured output schemas
from pydantic import BaseModel, SecretStr

# Securely load the Anthropic API key from Colab secrets
anthropic_api_key = SecretStr(userdata.get('ANTHROPIC_API_KEY'))

# Helper to pretty-print a model response with token usage statistics.
# Monitoring tokens is important because API costs are billed per token.
def print_response(response: AIMessage):
    print(f"Response id: {response.id}")
    if response.usage_metadata is not None:
        input_tokens = response.usage_metadata.get("input_tokens", 0)
        # Cache-read tokens were served from Anthropic's prompt cache (cheaper / faster)
        cached_tokens = response.usage_metadata.get("input_token_details", {}).get("cache_read", 0)
        output_tokens = response.usage_metadata.get("output_tokens", 0)
        # Reasoning tokens are consumed internally by extended-thinking models before producing output
        reasoning_tokens = response.usage_metadata.get("output_token_details", {}).get("reasoning", 0)

        print(f"Input tokens: {input_tokens} ({cached_tokens} cached); Output tokens: {output_tokens} ({reasoning_tokens} reasoning)")

    print()
    print(f"{'-' * 20} [Output] {'-' * 20}")
    print(response.text)


## Basic usage

In [ ]:
# Create a Claude model instance.
# "claude-haiku-4-5" is Anthropic's fast and cost-effective Claude model.
anthropic_default_model = ChatAnthropic(model="claude-haiku-4-5", api_key=anthropic_api_key)

# Send a two-message conversation to the model:
# - SystemMessage sets the model's role/persona
# - HumanMessage is the actual user request
museum_audio_guide_response = anthropic_default_model.invoke(
    input=[
        SystemMessage("You are a helpful history teaching assistant."),
        HumanMessage("Write a short museum audio-guide introduction for first-time visitors standing in front of the Rosetta Stone. Keep it under 5 sentences.")
    ]
)


In [ ]:
# Print the museum guide response with token usage stats
print_response(museum_audio_guide_response)


## Streaming

In [ ]:
# Demonstrate streaming — receive and print the model's response token by token.
# Streaming is useful for interactive UIs where you want to show output as it arrives
# rather than waiting for the full response.
# chunk.text contains the newly generated text for each streamed piece.
for chunk in anthropic_default_model.stream(
    input=[
        SystemMessage("You are an expert in culinary."),
        HumanMessage("Design a one-evening street-food route through Seoul for a curious first-time visitor who wants bold flavors but no seafood.")
    ]
):
    if chunk.text:
        # end="" prevents a newline after each chunk; flush=True sends it to the terminal immediately
        print(chunk.text, end="", flush=True)


## Multi-turn conversations

In [ ]:
# Demonstrate multi-turn (back and forth) conversation.
# The full conversation history is manually maintained in a list.
# Each new message is appended so the model always receives the entire context.
conversation = [
    # Tell the model to be concise and to remember details from earlier in the conversation
    SystemMessage("You are concise and remember user details from the chat history you receive."),
    HumanMessage("My name is Maria. I live in Plovdiv and I am preparing for a Python exam."),
]

# Turn 1: Send the opening message and get the first reply
first_reply = anthropic_default_model.invoke(conversation)
# Append the AI reply to the history so the next turn has full context
conversation.append(first_reply)


In [ ]:
# Print the model's first reply
print_response(first_reply)


In [ ]:
# Turn 2: Add a follow-up question to the existing conversation history.
# Because we pass the full history, the model remembers that the user is Maria from Plovdiv.
conversation.append(
    HumanMessage("What do you remember about me, and what should I focus on this week?")
)

second_reply = anthropic_default_model.invoke(conversation)
conversation.append(second_reply)


In [ ]:
# Print the second reply — the model should recall Maria's name, city, and exam preparation goal
print_response(second_reply)


## Reasoning

In [ ]:
# Enable extended thinking (reasoning) mode in Claude.
# With thinking enabled, the model spends extra "budget" tokens reasoning through the problem
# before producing its final answer. This improves quality for complex tasks.
# budget_tokens=4096 sets the maximum number of tokens Claude may spend on internal reasoning.
anthropic_thinking_model = ChatAnthropic(model="claude-haiku-4-5", api_key=anthropic_api_key, thinking={"type": "enabled", "budget_tokens": 4096})

book_cover_response = anthropic_thinking_model.invoke(
    input=[
        HumanMessage("Write a back-cover blurb for a literary novel about a family-run cinema trying to survive in the streaming era.")
    ]
)


In [ ]:
# Print the reasoning-enhanced book cover blurb.
# The token stats will show the reasoning_tokens consumed internally before the output.
print_response(book_cover_response)


## Structured output

In [ ]:
# Define a Pydantic model that describes the structure of the output we want.
# Pydantic models act as schemas: they specify field names, types, and validation rules.
# The model will be instructed to fill in this structure instead of returning free-form text.
class WorkshopBrief(BaseModel):
    title: str                      # Workshop title
    audience: str                   # Who the workshop is for
    duration_minutes: float         # How long the workshop lasts
    key_takeaways: list[str]        # Main things participants will learn
    materials_needed: list[str]     # Physical or digital items required

# with_structured_output() tells the model to respond with JSON that matches WorkshopBrief.
# LangChain then validates and parses that JSON into a WorkshopBrief Python object automatically.
anthropic_structured_output_model = anthropic_default_model.with_structured_output(WorkshopBrief)

workshop_brief = anthropic_structured_output_model.invoke(
    input=[
        SystemMessage("You are an expert event organizer."),
        HumanMessage("Design a beginner-friendly Saturday workshop about balcony herb gardening.")
    ]
)


In [ ]:
# Display the parsed WorkshopBrief object — a typed Python object, not raw JSON text.
# This is the power of structured output: the response is immediately usable in code.
workshop_brief


## Vision

<img src="https://freerangestock.com/sample/88947/painter-working-in-studio.jpg" />

In [ ]:
# Demonstrate vision (multi-modal) capability — send both text and an image to the model.
# Claude can analyze images alongside text in the same message.
# create_text_block(): wraps a plain text string as a message content block
# create_image_block(url=...): downloads the image from a URL and sends it to the model
analyze_image_response = anthropic_default_model.invoke(
    input=[
        SystemMessage("You are an expert image analyst. Keep your answer concise and structured."),
        HumanMessage(
            content_blocks=[
              # Instruction telling the model what to do with the image
              create_text_block("Analyze this image and return a one-sentence summary followed by 5 key visible objects."),
              # The actual image — Claude will "see" and describe this
              create_image_block(url="https://freerangestock.com/sample/88947/painter-working-in-studio.jpg")
            ]
        )
    ]
)


In [ ]:
# Print the image analysis result — a summary and list of visible objects from the painting image
print_response(analyze_image_response)
